In [86]:

import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
import pandas as pd

In [64]:
df = pd.read_csv('application_record.csv')

In [65]:
df.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0


In [66]:
df.columns

Index(['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN',
       'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH',
       'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE',
       'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS'],
      dtype='object')

In [67]:
df.shape

(438557, 18)

In [68]:
# Adding the Approval column
import pandas as pd
import numpy as np

def add_approval_column(df):
    score = pd.Series(50, index=df.index, dtype=float)

    # --- Income ---
    score += np.where(df['AMT_INCOME_TOTAL'] > 300000, 15,
             np.where(df['AMT_INCOME_TOTAL'] > 150000, 8,
             np.where(df['AMT_INCOME_TOTAL'] < 50000, -15, 0)))

    # --- Education ---
    score += np.where(df['NAME_EDUCATION_TYPE'].isin(['Higher education', 'Academic degree']), 12,
             np.where(df['NAME_EDUCATION_TYPE'] == 'Lower secondary', -10, 0))

    # --- Employment (DAYS_EMPLOYED is negative in raw data, convert to years) ---
    years_employed = df['DAYS_EMPLOYED'].abs() / 365
    score += np.where(years_employed > 5, 12,
             np.where(years_employed > 2, 6,
             np.where(years_employed < 1, -8, 0)))

    # --- Age (DAYS_BIRTH is negative in raw data, convert to years) ---
    age = df['DAYS_BIRTH'].abs() / 365
    score += np.where((age >= 30) & (age <= 55), 8,
             np.where(age < 22, -10, 0))

    # --- Income Type ---
    score += np.where(df['NAME_INCOME_TYPE'].isin(['State servant', 'Pensioner']), 8,
             np.where(df['NAME_INCOME_TYPE'] == 'Student', -12, 0))

    # --- Housing ---
    score += np.where(df['NAME_HOUSING_TYPE'] == 'House / apartment', 8,
             np.where(df['NAME_HOUSING_TYPE'] == 'With parents', -5, 0))

    # --- Assets ---
    score += np.where(df['FLAG_OWN_REALTY'] == 'Y', 10, 0)
    score += np.where(df['FLAG_OWN_CAR'] == 'Y', 5, 0)

    # --- Contact reliability ---
    score += np.where(df['FLAG_EMAIL'] == 1, 3, 0)
    score += np.where(df['FLAG_PHONE'] == 1, 2, 0)
    score += np.where(df['FLAG_WORK_PHONE'] == 1, 3, 0)

    # --- Children burden ---
    score += np.where(df['CNT_CHILDREN'] > 5, -18,
             np.where(df['CNT_CHILDREN'] > 3, -8, 0))

    # --- Income per family member ---
    income_per_member = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].replace(0, 1)
    score += np.where(income_per_member < 30000, -10, 0)

    # --- Occupation ---
    high_skill = ['Accountants', 'Managers', 'IT staff', 'Medicine staff', 'High skill tech staff']
    score += np.where(df['OCCUPATION_TYPE'].isin(high_skill), 8, 0)

    # --- Clip and threshold ---
    score = score.clip(0, 100)
    df['approved'] = (score >= 60).astype(int)

    return df


# Usage
df = pd.read_csv('application_record.csv')
df = add_approval_column(df)
print(df['approved'].value_counts())

approved
1    429848
0      8709
Name: count, dtype: int64


In [69]:
df.columns

Index(['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN',
       'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH',
       'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE',
       'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'approved'],
      dtype='object')

In [70]:
df = pd.get_dummies(df, columns=[
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE',
])
df = df.astype(int)

In [71]:
df.drop('ID', axis=1, inplace=True)

In [72]:
df.head()

,CNT_CHILDREN,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,approved,...,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff
0,0,427500,-12005,-4542,1,1,0,0,2,1,...,0,0,0,0,0,0,0,0,0,0
1,0,427500,-12005,-4542,1,1,0,0,2,1,...,0,0,0,0,0,0,0,0,0,0
2,0,112500,-21474,-1134,1,0,0,0,2,1,...,0,0,0,0,0,0,0,0,1,0
3,0,270000,-19110,-3051,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0
4,0,270000,-19110,-3051,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0


In [73]:
df.shape

(438557, 55)

In [74]:
df['DAYS_BIRTH'] = abs(df['DAYS_BIRTH']) / 365
df['DAYS_EMPLOYED'] = abs(df['DAYS_EMPLOYED']) / 365

In [75]:
df.head()

,CNT_CHILDREN,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,approved,...,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff
0,0,427500,32.890411,12.443836,1,1,0,0,2,1,...,0,0,0,0,0,0,0,0,0,0
1,0,427500,32.890411,12.443836,1,1,0,0,2,1,...,0,0,0,0,0,0,0,0,0,0
2,0,112500,58.832877,3.106849,1,0,0,0,2,1,...,0,0,0,0,0,0,0,0,1,0
3,0,270000,52.356164,8.358904,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0
4,0,270000,52.356164,8.358904,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0


In [76]:
df.rename(columns={
    'DAYS_BIRTH':'AGE',
    'DAYS_EMPLOYED':'EMPLOYMENT_YEARS'
}, inplace=True)

In [77]:
df.head()

,CNT_CHILDREN,AMT_INCOME_TOTAL,AGE,EMPLOYMENT_YEARS,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,approved,...,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff
0,0,427500,32.890411,12.443836,1,1,0,0,2,1,...,0,0,0,0,0,0,0,0,0,0
1,0,427500,32.890411,12.443836,1,1,0,0,2,1,...,0,0,0,0,0,0,0,0,0,0
2,0,112500,58.832877,3.106849,1,0,0,0,2,1,...,0,0,0,0,0,0,0,0,1,0
3,0,270000,52.356164,8.358904,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0
4,0,270000,52.356164,8.358904,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0


In [78]:
X = df.drop(['approved'],axis=1)
y = df['approved']

In [83]:
#Normallization

# Re-create X from the one-hot encoded and integer-typed df
# This ensures all columns in X are numerical before normalization

X_min = X.min()
X_max = X.max()
# Add a small epsilon to avoid division by zero in case max_val == min_val for any column
X_scaled = (X - X_min) / (X_max - X_min + 1e-8)

In [84]:
X_scaled

,CNT_CHILDREN,AMT_INCOME_TOTAL,AGE,EMPLOYMENT_YEARS,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,CODE_GENDER_F,...,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff
0,0.0,0.059697,0.254968,0.012403,0.0,1.0,0.0,0.0,0.052632,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.059697,0.254968,0.012403,0.0,1.0,0.0,0.0,0.052632,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.012850,0.789578,0.003072,0.0,0.0,0.0,0.0,0.052632,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.0,0.036274,0.656109,0.008321,0.0,0.0,1.0,1.0,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.036274,0.656109,0.008321,0.0,0.0,1.0,1.0,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438552,0.0,0.016196,0.859756,1.000000,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
438553,0.0,0.011511,0.477078,0.008200,0.0,0.0,0.0,0.0,0.000000,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
438554,0.0,0.004149,0.038392,0.000986,0.0,1.0,0.0,0.0,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
438555,0.0,0.006826,0.800813,1.000000,0.0,0.0,0.0,0.0,0.052632,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [85]:
X_train, X_test, y_train, y_test = train_test_split(
X_scaled, y, test_size=0.33, random_state=42,stratify=y)

In [96]:
model_ANN = keras.Sequential([
    layers.Input(shape = (X_scaled.shape[1],)),
    layers.Dense(32,activation='relu'),
    layers.Dense(16,activation='relu'),
    layers.Dense(1,activation='sigmoid')
])

In [97]:
from sklearn import metrics
model_ANN.compile(optimizer='sgd',loss='binary_crossentropy',metrics=['accuracy'])

In [99]:
history = model_ANN.fit(
    X_train.values,y_train.values,
    validation_data = (X_test.values,y_test.values),
    epochs = 100,batch_size = 1000 , verbose = 1
)

Epoch 1/100
294/294 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9804 - loss: 0.0651 - val_accuracy: 0.9806 - val_loss: 0.0632
Epoch 2/100
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9810 - loss: 0.0620 - val_accuracy: 0.9806 - val_loss: 0.0629
Epoch 3/100
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9803 - loss: 0.0633 - val_accuracy: 0.9807 - val_loss: 0.0628
Epoch 4/100
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9805 - loss: 0.0624 - val_accuracy: 0.9807 - val_loss: 0.0626
Epoch 5/100
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9804 - loss: 0.0632 - val_accuracy: 0.9807 - val_loss: 0.0624
Epoch 6/100
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9807 - loss: 0.0620 - val_accuracy: 0.9808 - val_loss: 0.0623
Epoch 7/100
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9810 - loss: 0.0613 - val_accuracy: 0.9808 - val_loss: 0.0622
Epoch 8/100
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9808 - loss: 0.0615 - val_accu

In [100]:
import joblib

# save ANN model
model_ANN.save("ann_model.h5")

# save normalization values
joblib.dump(X_min, "X_min.pkl")
joblib.dump(X_max, "X_max.pkl")

# save feature columns
joblib.dump(X.columns.tolist(), "columns.pkl")

['columns.pkl']